# PANOSETI Topology Engine Demo

This notebook demonstrates the PANOSETI topology engine, featuring hub-spoke layouts, interactive visualization, and subnet coherence validation.

In [51]:
import os
import sys
from unittest.mock import MagicMock

from rich import print

# Add src to path
# sys.path.insert(0, os.path.abspath("src"))
from control.topology.fleet import generate_fleet_configs
from control.topology.graph_builder import GraphBuilder
from control.topology.visualizer import (
    export_interactive_html,
    save_topology_image,
)
from control.utils.global_validator import GlobalConfigValidator
from control.utils.paths import PanoPaths

## 1. Generate Fleet & Build Graph

We'll create a complex fleet with multiple modules and randomized subnets.

In [58]:
daq_config, quabo_uids = generate_fleet_configs(
    num_daq_nodes=7, 
    modules_per_node=2, 
    subnet_probability=1.0
)

builder = GraphBuilder()
graph = builder.build_from_configs(daq_config, quabo_uids)

print(f"Topology ready with {graph.number_of_nodes()} nodes.")

Topology ready with 85 nodes.

## 3. Interactive Topology

Interactive HTML export using Pyvis, saved to the transient workspace.

In [59]:
html_path = "topology_notebook.html"
export_interactive_html(graph, html_path)

from IPython.display import IFrame

IFrame(src=str(html_path), width="80%", height="800px")

## 4. Subnet Coherence Validation

Verify that DAQ nodes and their Quabos are in the same subnet to ensure optimal data rates.

In [57]:
# Mock network config for validation demo
mock_net = MagicMock()
mock_net.daq_nodes = []
mock_net.modules = []

configs = {
    'daq': daq_config,
    'obs': MagicMock(domes=quabo_uids.domes),
    'network': mock_net, # Simplified for demo
    'firmware': None,
    'data': None
}

validator = GlobalConfigValidator(configs)
validator._check_daq_module_subnet_coherence()

validator.report.print_report()

                          Global Tier-2 Validation Report                           
┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Test Name        ┃ Status ┃ Details                                              ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ Subnet Coherence │  PASS  │ All DAQ nodes and Quabos reside in coherent subnets. │
└──────────────────┴────────┴──────────────────────────────────────────────────────┘